In [16]:
import os
import urllib
import tarfile
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets
import torchvision.transforms as T
import matplotlib.pyplot as plt
# from torchvision.datasets import ImageNet # The dataset from the competition AlexNet was specifically made for

In [3]:
torch.manual_seed(41)

In [4]:
train_transforms = T.Compose([
    T.Resize((256,256)), # Standard ImageNet rescaling
    T.RandomCrop(224), # Random cropping (224x224)
    T.RandomHorizontalFlip(p=0.5),
    T.ToTensor(), # Transforms the PIL image into Tensor (0.0 - 1.0)
    T.Normalize(mean=[0.485, 0.456, 0.406],
                 std=[0.229, 0.224, 0.225]
    ),
])

val_transforms = T.Compose([
    T.Resize((256,256)),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
"""Folder location where both splits of the Imagenette dataset are located

Source: https://github.com/fastai/imagenette
Install Link (Full size): https://s3.amazonaws.com/fast-ai-imageclas/imagenette2.tgz
"""
extract_dir='data'

In [ ]:
"""ImageFolder instances so that we know the location of both the train and validation splits and link the corresponding transformations"""
train_dataset = datasets.ImageFolder(root=os.path.join(extract_dir, 'train'), transform=train_transforms)
val_dataset = datasets.ImageFolder(root=os.path.join(extract_dir, 'val'), transform=val_transforms)

NameError: name 'extract_dir' is not defined

In [ ]:
"""AlexNet standard batch size"""
BATCH_SIZE=128

In [ ]:
"""Instantiate the DataLoaders for both the train and val split"""
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
"""Visualizing an example image batch from the imagenette dataset"""
rand_idx = torch.randint(0, BATCH_SIZE, (1,)).item() # Generates random numbers from 0 to BATCH_SIZE-1; the (1,) parameter basically indicates pytorch to return a tensor with a single element
train_features, train_labels = next(iter(train_loader))
print(f"Feature batch shape: {train_features.size()}")
print(f"Labels batch shape: {train_labels.size()}")
print(train_features[0].shape())
print(train_features[0].squeeze().shape())
img = train_features[0].squeeze() # [B, H, W] -> [H, W]
label = train_labels[0]
plt.imshow(img, cmap='gray')
plt.title("Here is the first image from the first batch")
plt.show()
print(f"Label: {label}")

# We can train AlexNet on ImageNet in 2 ways:
- Using the class method
- Manually iterating through the train_loader 

## A.

In [ ]:
from src.model import AlexNet

model = AlexNet()
model.fit(train_loader=train_loader, val_loader=val_loader, device='cpu')

## B. -> Includes loss visualization

In [ ]:
EPOCHS=90
loss_l = []
device = 'cuda' if torch.cuda.is_available() else 'cpu'
optimizer = torch.optim.SGD(model.parameters(), momentum=0.9, weight_decay=0.0005, lr=0.01)
criterion = nn.CrossEntropyLoss()
for epoch in range(EPOCHS):  
    running_loss = 0.0
    for batch_idx, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs=model(inputs)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        loss_l.append(loss.item())
        running_loss += loss.item()

        if batch_idx % 100 == 0:
            print(f"Epoch: {epoch} | Batch: {batch_idx:03d} | Batch Loss: {loss.item():.4f}")

    epoch_loss = running_loss / len(train_loader)
    
    print(f"Epoch {epoch} completed | Average Loss: {epoch_loss:.4f}")


In [ ]:
"""Mostly boilerplate/boring matplotlib code"""
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(loss_l, label='Loss', color='blue', linewidth=1)

ax.set_title("Loss Evolution during AlexNet Training")
ax.set_xlabel("Iterations")
ax.set_ylabel('Loss Value')
ax.grid(True, linestyle='--', alpha=0.6)
for i in range(EPOCHS):
    ax.annotate(f'Epoch {i}', xy=(i-1, loss_l[i-1]), arrowprops=dict(facecolor='red', arrowstyle='-|>', edgecolor='black', shrink=0.05))
ax.legend()
plt.show()


# Now for evaluation of the AlexNet model

## A.

In [ ]:
model.evaluate(val_loader=val_loader, device='cpu', verbose=True)

## B.

In [ ]:
"""Validation split evaluation"""
model.eval()

correct = 0
predicted = 0
y_true = []
y_pred = []

with torch.no_grad(): # Disable gradient computation
    for images, labels in val_loader:
        outputs = model(images)
        
        predicted = torch.argmax(outputs, dim=1) # 
        y_pred.extend(predicted.cpu().numpy())
        y_true.extend(labels.cpu().numpy())
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
accuracy = 100 * correct / total
print(f"Accuracy of the model on the test set: {accuracy:.2f}%")